In [ ]:
import os
import subprocess

REPO = "/kaggle/working/ReplicateTest"
if not os.path.isdir(REPO):
    subprocess.run(
        ["git", "clone", "https://github.com/xieyangyuyue/ReplicateTest.git", REPO],
        check=True)
else:
    print("仓库已存在，跳过 clone")

%cd /kaggle/working/ReplicateTest/semantic-segmentation
!ls


In [ ]:
# ============================================================
# W&B 登录（项目 ReplicateTest）
# Kaggle：Add-ons -> Secrets 添加 WANDB_API_KEY，切勿把 key 明文写进 notebook
# 登录一次后，torchrun 的子进程会复用 ~/.netrc
# ============================================================
!pip install -q wandb

import os
import wandb

api_key = None
try:
    from kaggle_secrets import UserSecretsClient
    api_key = UserSecretsClient().get_secret("WANDB_API_KEY")
except Exception as exc:
    print("Kaggle Secrets 读取失败：", exc)

api_key = api_key or os.environ.get("WANDB_API_KEY")
if not api_key:
    raise RuntimeError("请在 Kaggle Secrets 中配置 WANDB_API_KEY")

wandb.login(key=api_key)
print("W&B 登录成功")


In [ ]:
# ============================================================
# 数据准备：下载 VOC2012，并确保 datasets/voc 指向数据目录
# 若已挂载 Kaggle Dataset，把 VOC_SRC 改成对应路径即可（含 VOC2012/ 子目录）
# ============================================================
import os
import subprocess

PROJECT_ROOT = os.getcwd()
VOC_SRC = os.path.expanduser("~/.torch/datasets/VOCdevkit")
VOC_LINK = os.path.join(PROJECT_ROOT, "datasets", "voc")

if not os.path.isdir(os.path.join(VOC_SRC, "VOC2012")):
    subprocess.run(
        ["python", "-m", "core.data.downloader.pascal_voc", "--download-dir", VOC_SRC],
        check=True)
else:
    print("VOC2012 已存在，跳过下载")

# downloader 通常已经建好链接；这里做一次修复，保证链接一定有效
os.makedirs(os.path.dirname(VOC_LINK), exist_ok=True)
if os.path.islink(VOC_LINK) or os.path.isfile(VOC_LINK):
    os.remove(VOC_LINK)
elif os.path.isdir(VOC_LINK) and os.listdir(VOC_LINK):
    raise RuntimeError("{} 是非空真实目录，请手动移走后再建链接".format(VOC_LINK))
elif os.path.isdir(VOC_LINK):
    os.rmdir(VOC_LINK)
os.symlink(os.path.realpath(VOC_SRC), VOC_LINK)
print("数据集：", VOC_LINK, "->", os.path.realpath(VOC_LINK))


In [ ]:
# 环境检查：确认 Kaggle 分配到的 GPU
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
!nvidia-smi -L
!nproc


In [ ]:
%cd /kaggle/working/ReplicateTest/semantic-segmentation/scripts

# ============================================================
# 训练参数：全部集中在这里，按注释调整后直接运行本 cell
# ------------------------------------------------------------
# MODEL/BACKBONE : 原实验 FCN32s+VGG16；想要更省显存的 stride-8 FCN，
#                  改成 MODEL="fcn", BACKBONE="resnet50"
# LR             : 每卡基准学习率，train.py 会按 GPU 数线性放大（2 卡 => 2e-4）
# BATCH_SIZE     : 每卡训练批大小（T4 16GB + AMP 下 4 是稳妥值）
# VAL_BATCH_SIZE : 每卡验证批大小；验证不做反传，可调大来提升验证时双卡利用率
# WORKERS        : 每个 DDP 进程的 DataLoader worker 数；Kaggle ~4 vCPU，
#                  2 卡 × 2 worker 已足够，设 4 反而抢 CPU 变慢
# AMP_FLAG       : T4 支持 FP16，省显存且更快；不需要时改成 ""
# BEST_METRIC    : 按哪个验证指标保存 best_model
# 其余默认：每 1 个 epoch 验证一次、每 10 个 epoch 存一次 ckpt；
#           训练/验证进度条实时刷新，日志按 epoch 落盘
# 若 VGG16 权重上传太慢，在下面命令末尾加 --no-wandb-save-model
# ============================================================
MODEL = "fcn32s"          # 可选 "fcn"（ResNet FCN）、"fcn16s"、"fcn8s"
BACKBONE = "vgg16"        # MODEL="fcn" 时改成 "resnet50"
DATASET = "pascal_voc"
EPOCHS = 100
LR = 1e-4
BATCH_SIZE = 4
VAL_BATCH_SIZE = 8
WORKERS = 2
AMP_FLAG = "--amp"
BEST_METRIC = "miou"      # miou / pixacc / mpa / fwiou / mean
NPROC = 2                 # Kaggle T4 x2
MASTER_PORT = 29500

RUN_ROOT = "/kaggle/working/runs"   # 只改这里即可整体换输出位置
SAVE_DIR = RUN_ROOT + "/checkpoints"
LOG_DIR = RUN_ROOT + "/logs"
WANDB_PROJECT = "ReplicateTest"

!mkdir -p {SAVE_DIR} {LOG_DIR}

# run 名由 train.py 自动生成：模型-骨干-数据集-ddpN-lr-bs-ep-amp/fp32-时间戳
# 同配置多次运行不会重名；W&B 中按 group + tags + config 区分不同实验
!torchrun --nproc_per_node={NPROC} --master_port={MASTER_PORT} train.py \
    --model {MODEL} --backbone {BACKBONE} --dataset {DATASET} \
    --lr {LR} --epochs {EPOCHS} \
    --batch-size {BATCH_SIZE} --val-batch-size {VAL_BATCH_SIZE} \
    --workers {WORKERS} {AMP_FLAG} \
    --best-metric {BEST_METRIC} \
    --save-dir {SAVE_DIR} --log-dir {LOG_DIR} \
    --wandb --wandb-project {WANDB_PROJECT}


In [ ]:
# ============================================================
# 保存输出：模型 + runs/log
# train.py 已把 best_model 与日志作为 W&B Artifact 上传；
# 这里再打包一份，方便从 Kaggle Output 直接下载
# ============================================================
import glob
import os
import shutil

for path in glob.glob("/kaggle/working/runs/checkpoints/*_best_model.pth"):
    shutil.copy(path, "/kaggle/working/")
    print("copied:", os.path.basename(path))

!cd /kaggle/working && rm -f runs.zip && zip -r -q runs.zip runs
!ls -lh /kaggle/working/runs.zip
!find /kaggle/working/runs -maxdepth 2 -type f | head -30
